# LeetCode #7 — Reverse Integer: đảo chữ số và kiểm tra tràn số 32-bit

**Mức độ:** Medium · **Chủ đề:** Math

Skeleton trên LeetCode:

```python
class Solution:
    def reverse(self, x: int) -> int:
```

Ở bài Palindrome Number, bạn đã học cách đảo một số bằng `% 10` và `// 10`. Bài này dùng lại đúng kỹ thuật đó, nhưng thêm hai thử thách mới:

1. **Số âm** — và cách Python xử lý `%`, `//` với số âm rất khác C/Java.
2. **Tràn số (overflow)** — kết quả phải nằm trong phạm vi số nguyên 32-bit có dấu, và ta phải phát hiện tràn **trước khi** nó xảy ra.

---

## Sau notebook này, bạn sẽ nắm được

1. Phạm vi số nguyên 32-bit có dấu là gì, vì sao là `[-2^31, 2^31 - 1]`.
2. Công thức đảo số `rev = rev * 10 + digit` qua ví dụ cụ thể.
3. Cái bẫy `%` và `//` với số âm trong Python.
4. Cách giải bằng chuỗi và vì sao nó chưa đúng tinh thần đề.
5. Cách **kiểm tra tràn trước khi nhân** — kỹ thuật dùng lại ở bài #8 String to Integer.
6. Bài tập vận dụng từ dễ đến nâng cao.

# 1. Bài toán là gì?

Cho một số nguyên 32-bit có dấu `x`. Hãy trả về `x` với **các chữ số bị đảo ngược**.

Nếu kết quả đảo **vượt ra ngoài** phạm vi `[-2^31, 2^31 - 1]`, trả về `0`.

Đề còn ghi chú: **giả sử môi trường không cho phép lưu số nguyên 64-bit** (có dấu hay không dấu).

**Constraints (theo đề):** `-2^31 <= x <= 2^31 - 1`

---

## Ví dụ

```text
x = 123   ->  321
x = -123  ->  -321
x = 120   ->  21        (số 0 ở đầu "021" biến mất)
```

Thêm một ví dụ tràn số:

```text
x = 1534236469  ->  đảo thành 9646324351, lớn hơn 2^31 - 1  ->  trả về 0
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Số nguyên 32 bit, tràn số và tách chữ số
2. Tính toán theo từng chữ số (cộng / nhân có nhớ)

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Số nguyên 32 bit, tràn số và tách chữ số

### Giới hạn 32 bit có dấu

Nhiều đề yêu cầu "môi trường chỉ lưu được số nguyên 32 bit có dấu":

```text
INT_MIN = -2^31     = -2147483648
INT_MAX =  2^31 - 1 =  2147483647
```

Vượt khỏi khoảng này gọi là **tràn số** (overflow). C/Java sẽ cho kết quả sai (quay vòng); Python thì số nguyên **không giới hạn**, nên ta phải **tự kiểm tra** để đúng yêu cầu đề.

In [1]:
INT_MIN_D, INT_MAX_D = -2 ** 31, 2 ** 31 - 1
print(INT_MIN_D, INT_MAX_D)
print("Python không tràn:", INT_MAX_D + 1, "| mô phỏng C (quay vòng):", ((INT_MAX_D + 1 + 2 ** 31) % 2 ** 32) - 2 ** 31)

def kep_32(x):
    return max(INT_MIN_D, min(INT_MAX_D, x))
print("kẹp 10^10 về:", kep_32(10 ** 10), "| kẹp -10^10:", kep_32(-10 ** 10))

-2147483648 2147483647
Python không tràn: 2147483648 | mô phỏng C (quay vòng): -2147483648
kẹp 10^10 về: 2147483647 | kẹp -10^10: -2147483648


### Tách và ghép chữ số

- Chữ số cuối: `x % 10`; bỏ chữ số cuối: `x // 10` (với `x >= 0`).
- Ghép thêm chữ số `d` vào bên phải: `x = x * 10 + d`.
- `divmod(x, 10)` trả cả hai cùng lúc.

**Cẩn thận với số âm:** trong Python `-123 // 10 = -13` và `-123 % 10 = 7` (làm tròn **xuống**). Nên xử lý **dấu riêng**, làm việc với `abs(x)`.

In [2]:
x_d = 12345
chu_so = []
while x_d > 0:
    x_d, d = divmod(x_d, 10)
    chu_so.append(d)
print("chữ số (từ phải sang):", chu_so)
y_d = 0
for d in [5, 4, 3, 2, 1]:
    y_d = y_d * 10 + d
print("ghép lại:", y_d)
print("số âm: -123 // 10 =", -123 // 10, "| -123 % 10 =", -123 % 10, "| int(-123 / 10) =", int(-123 / 10))

chữ số (từ phải sang): [5, 4, 3, 2, 1]
ghép lại: 54321
số âm: -123 // 10 = -13 | -123 % 10 = 7 | int(-123 / 10) = -12


### Kiểm tra tràn **trước khi** tràn

Khi không được dùng số lớn hơn 32 bit (như trong C), phải kiểm tra trước bước `x = x * 10 + d`:

```text
x * 10 + d > INT_MAX   ⟺   x > INT_MAX // 10   hoặc   (x == INT_MAX // 10 và d > 7)
```

(vì `INT_MAX = 214748364·10 + 7`). Trong Python có thể tính rồi so sánh, nhưng hiểu điều kiện này giúp viết đúng ở mọi ngôn ngữ.

In [3]:
def ghep_an_toan(chu_so_list):
    x = 0
    for d in chu_so_list:
        if x > INT_MAX_D // 10 or (x == INT_MAX_D // 10 and d > 7):
            return None                                   # sẽ tràn
        x = x * 10 + d
    return x

print(ghep_an_toan([2, 1, 4, 7, 4, 8, 3, 6, 4, 7]), ghep_an_toan([2, 1, 4, 7, 4, 8, 3, 6, 4, 8]))

2147483647 None


## 2.2 Tính toán theo từng chữ số (cộng / nhân có nhớ)

### Cộng tay có nhớ

Ta cộng từ **hàng đơn vị** sang trái; mỗi cột: `tổng = a + b + nhớ`, ghi `tổng % 10`, nhớ `tổng // 10` sang cột kế.

```text
      1 1         ← nhớ
      4 8 7
  +   3 6 5
  ---------
      8 5 2
```

In [4]:
def cong_chuoi(x, y):
    i, j, nho, out = len(x) - 1, len(y) - 1, 0, []
    while i >= 0 or j >= 0 or nho:
        a_ = int(x[i]) if i >= 0 else 0
        b_ = int(y[j]) if j >= 0 else 0
        tong = a_ + b_ + nho
        out.append(str(tong % 10))
        nho = tong // 10
        print(f"   cột: {a_} + {b_} + nhớ → {tong}: ghi {tong % 10}, nhớ {nho}")
        i -= 1; j -= 1
    return "".join(reversed(out))

print("487 + 365 =", cong_chuoi("487", "365"))
print("999 + 1 =", cong_chuoi("999", "1"), "(nhớ ở cột cuối tạo thêm chữ số)")

   cột: 7 + 5 + nhớ → 12: ghi 2, nhớ 1
   cột: 8 + 6 + nhớ → 15: ghi 5, nhớ 1
   cột: 4 + 3 + nhớ → 8: ghi 8, nhớ 0
487 + 365 = 852
   cột: 9 + 1 + nhớ → 10: ghi 0, nhớ 1
   cột: 9 + 0 + nhớ → 10: ghi 0, nhớ 1
   cột: 9 + 0 + nhớ → 10: ghi 0, nhớ 1
   cột: 0 + 0 + nhớ → 1: ghi 1, nhớ 0
999 + 1 = 1000 (nhớ ở cột cuối tạo thêm chữ số)


### Nhân tay: chữ số hàng `i` × chữ số hàng `j` rơi vào hàng `i + j`

Đánh số từ **phải** (hàng đơn vị là 0): tích hai chữ số ở hàng `i` và `j` đóng góp vào hàng `i + j` của kết quả. Cộng dồn mọi đóng góp rồi xử lý nhớ một lượt. Tích của số `m` chữ số và `n` chữ số có tối đa `m + n` chữ số.

```text
        1 2 3          hàng:  2 1 0
      ×   4 5                   1 0
      -------
        6 1 5          123 × 5
      4 9 2            123 × 4, dịch một hàng
      -------
      5 5 3 5
```

In [5]:
def nhan_chuoi(x, y):
    cot = [0] * (len(x) + len(y))           # cot[k] = tổng đóng góp vào hàng k (từ phải)
    for i, dx in enumerate(reversed(x)):
        for j, dy in enumerate(reversed(y)):
            cot[i + j] += int(dx) * int(dy)
    nho = 0
    for k in range(len(cot)):               # xử lý nhớ
        nho, cot[k] = divmod(cot[k] + nho, 10)
    s = "".join(map(str, reversed(cot))).lstrip("0")
    return s or "0"

print("123 × 45 =", nhan_chuoi("123", "45"), "| kiểm tra:", 123 * 45)
print("0 × 999 =", nhan_chuoi("0", "999"))

123 × 45 = 5535 | kiểm tra: 5535
0 × 999 = 0


Lỗi hay gặp: quên **nhớ ở cột cuối** (`999 + 1`); đọc chữ số từ trái trong khi phải cộng từ phải; quên bỏ số 0 đầu (`"00"` → `"0"`); hai số dài khác nhau — hết chữ số thì coi là 0.

## 2.3 Áp dụng vào bài này

Đảo chữ số chính là **tách chữ số** bằng `% 10`, `// 10` rồi **ghép** lại bằng `x * 10 + d`. Điểm khó là giới hạn **32 bit có dấu**: kết quả vượt `[−2^31, 2^31 − 1]` phải trả `0`, và cần xử lý **dấu riêng** vì `//`, `%` của Python với số âm làm tròn xuống.

# 3. Số nguyên 32-bit có dấu

Máy tính lưu số nguyên trong một số **bit** cố định. Với 32 bit có dấu:

- 1 bit dành cho dấu, 31 bit cho độ lớn.
- Nhỏ nhất: `-2^31 = -2 147 483 648`
- Lớn nhất: `2^31 - 1 = 2 147 483 647`

Phía âm "rộng hơn" phía dương đúng 1 đơn vị.

Python thì **không có giới hạn** này: `int` của Python lớn bao nhiêu cũng được. Vì vậy ta phải **tự kiểm tra** phạm vi.

In [6]:
INT_MIN = -2**31
INT_MAX = 2**31 - 1
print("INT_MIN =", INT_MIN)
print("INT_MAX =", INT_MAX)

print(9646324351 > INT_MAX)      # True -> tràn
print(2**100)                    # Python vẫn tính được, không tràn

INT_MIN = -2147483648
INT_MAX = 2147483647
True
1267650600228229401496703205376


# 4. Nhắc lại: tách chữ số bằng `% 10` và `// 10`

Với số **dương** `x`:

- `x % 10` lấy chữ số **cuối** (hàng đơn vị).
- `x // 10` **bỏ** chữ số cuối.

In [7]:
x = 123
print(x % 10)    # 3
print(x // 10)   # 12

x = 12
print(x % 10)    # 2
print(x // 10)   # 1

3
12
2
1


## Công thức đảo số

Ta "bóc" từng chữ số từ cuối `x`, và "đẩy" nó vào cuối `rev`:

```python
digit = x % 10
x = x // 10
rev = rev * 10 + digit
```

`rev * 10` dịch các chữ số đã có sang trái một hàng, chừa chỗ cho chữ số mới ở hàng đơn vị.

Với `x = 123`:

| Bước | digit | x sau | rev = rev * 10 + digit |
|---|---|---|---|
| 1 | 3 | 12 | 0 * 10 + 3 = 3 |
| 2 | 2 | 1 | 3 * 10 + 2 = 32 |
| 3 | 1 | 0 | 32 * 10 + 1 = 321 |

Dừng khi `x == 0`.

In [8]:
x = 123
rev = 0
while x != 0:
    digit = x % 10
    x //= 10
    rev = rev * 10 + digit
    print(f"digit = {digit}, x = {x}, rev = {rev}")

digit = 3, x = 12, rev = 3
digit = 2, x = 1, rev = 32
digit = 1, x = 0, rev = 321


Với `x = 120`: chữ số đầu tiên bóc ra là `0`, nên `rev = 0 * 10 + 0 = 0` — số 0 thừa tự biến mất. Không cần xử lý riêng.

In [9]:
x = 120
rev = 0
while x != 0:
    digit = x % 10
    x //= 10
    rev = rev * 10 + digit
    print(f"digit = {digit}, x = {x}, rev = {rev}")

digit = 0, x = 12, rev = 0
digit = 2, x = 1, rev = 2
digit = 1, x = 0, rev = 21


# 5. Cái bẫy: `%` và `//` với số âm trong Python

Trong Python, `//` là **chia làm tròn xuống** (về phía âm vô cùng), và `%` luôn có **cùng dấu với số chia** (ở đây là `10`, dương).

In [10]:
print(-123 // 10)   # -13, không phải -12!
print(-123 % 10)    # 7,   không phải -3!
print(-13 * 10 + 7) # -123: Python luôn đảm bảo (x // 10) * 10 + x % 10 == x

-13
7
-123


Trong C/Java, `-123 / 10 = -12` và `-123 % 10 = -3` (làm tròn **về 0**). Nhiều lời giải trên mạng viết cho C/Java, chép nguyên sang Python sẽ chạy sai với số âm — thậm chí lặp vô hạn, vì `x // 10` với `x = -1` cho ra `-1` mãi mãi.

In [11]:
print(-1 // 10)     # -1: nếu dùng while x != 0 với số âm, vòng lặp không bao giờ dừng

-1


## Cách an toàn: tách dấu ra trước

```python
sign = -1 if x < 0 else 1
x = abs(x)
# ... đảo số dương x ...
result = sign * rev
```

Đảo `-123` = đảo `123` rồi gắn dấu âm = `-321`.

# 6. Cách dễ nghĩ nhất: dùng chuỗi

1. Lấy dấu.
2. Đổi `abs(x)` thành chuỗi, đảo bằng `[::-1]`, đổi lại thành số.
3. Gắn dấu, kiểm tra phạm vi.

In [12]:
def reverse_string(x: int) -> int:
    sign = -1 if x < 0 else 1
    rev = int(str(abs(x))[::-1])   # "120" -> "021" -> 21
    result = sign * rev
    if result < INT_MIN or result > INT_MAX:
        return 0
    return result

for x in [123, -123, 120, 0, 1534236469]:
    print(f"{x:>11} -> {reverse_string(x)}")

        123 -> 321
       -123 -> -321
        120 -> 21
          0 -> 0
 1534236469 -> 0


## Vấn đề của cách này

Trong Python, cách trên **cho kết quả đúng**. Nhưng nó vi phạm tinh thần đề:

- Ta tạo ra số `9646324351` rồi mới kiểm tra. Trong môi trường **chỉ có 32-bit**, số này **không thể tồn tại** — nó đã bị tràn từ trước.
- Trong phỏng vấn, người ta muốn thấy bạn **phát hiện tràn trước khi nó xảy ra**.

Câu hỏi: làm sao biết `rev * 10 + digit` sẽ vượt `INT_MAX` **mà không cần tính nó**?

# 7. Ý tưởng then chốt: kiểm tra tràn trước khi nhân

Ta muốn đảm bảo:

```text
rev * 10 + digit <= INT_MAX = 2147483647
```

Chia hai vế cho 10, so sánh phần nguyên:

- `INT_MAX // 10 = 214748364`
- `INT_MAX % 10  = 7`

Có ba khả năng:

| Điều kiện | Kết luận |
|---|---|
| `rev < 214748364` | `rev * 10 <= 2147483630`, cộng `digit ≤ 9` vẫn ≤ `2147483639` → **an toàn** |
| `rev > 214748364` | `rev * 10 >= 2147483650` → **chắc chắn tràn** |
| `rev == 214748364` | an toàn khi và chỉ khi `digit <= 7` |

Mọi phép so sánh ở đây chỉ dùng số **nhỏ hơn** `INT_MAX`, nên không cần 64-bit.

## Còn phía âm?

Ta làm việc với `abs(x)`, nên phía âm có giới hạn độ lớn là `2^31 = 2147483648` (chữ số cuối `8`). Gọn nhất là đặt:

```python
limit = INT_MAX if sign > 0 else 2**31       # độ lớn tối đa cho phép
```

rồi dùng cùng một phép kiểm tra với `limit // 10` và `limit % 10`.

*Lưu ý tinh tế:* trong Python, bản thân `2**31` vượt 32-bit dương. Ở C/Java, người ta thường xử lý trên số âm để tránh điều này. Trong notebook, ta giữ cách `limit` vì dễ hiểu; logic kiểm tra vẫn đúng y hệt.

In [13]:
limit = INT_MAX
print("limit // 10 =", limit // 10, ", limit % 10 =", limit % 10)

def will_overflow(rev, digit, limit):
    return rev > limit // 10 or (rev == limit // 10 and digit > limit % 10)

print(will_overflow(214748363, 9, INT_MAX))   # False: 2147483639 an toàn
print(will_overflow(214748364, 7, INT_MAX))   # False: 2147483647 vừa đúng INT_MAX
print(will_overflow(214748364, 8, INT_MAX))   # True:  2147483648 tràn
print(will_overflow(214748365, 0, INT_MAX))   # True
print(will_overflow(214748364, 8, 2**31))     # False: phía âm cho phép -2147483648

limit // 10 = 214748364 , limit % 10 = 7
False
False
True
True
False


# 8. Lời giải chính

In [14]:
def reverse_integer(x: int) -> int:
    sign = -1 if x < 0 else 1
    x = abs(x)
    limit = INT_MAX if sign > 0 else 2**31

    rev = 0
    while x != 0:
        digit = x % 10
        x //= 10

        if rev > limit // 10 or (rev == limit // 10 and digit > limit % 10):
            return 0                     # sắp tràn -> dừng ngay

        rev = rev * 10 + digit

    return sign * rev

for x in [123, -123, 120, 0, 1534236469]:
    print(f"{x:>11} -> {reverse_integer(x)}")

        123 -> 321
       -123 -> -321
        120 -> 21
          0 -> 0
 1534236469 -> 0


# 9. Mô phỏng từng bước

## Trường hợp tràn: `x = 1534236469`

| Bước | digit | rev trước | So với 214748364 | Hành động | rev sau |
|---|---|---|---|---|---|
| 1 | 9 | 0 | nhỏ hơn | đẩy | 9 |
| 2 | 6 | 9 | nhỏ hơn | đẩy | 96 |
| … | … | … | … | … | … |
| 9 | 5 | 96463243 | nhỏ hơn | đẩy | 964632435 |
| 10 | 1 | 964632435 | **lớn hơn** | **trả về 0** | — |

Ở bước 10, ta phát hiện tràn **trước khi** tính `964632435 * 10`.

In [15]:
def trace_reverse(x: int) -> int:
    sign = -1 if x < 0 else 1
    x = abs(x)
    limit = INT_MAX if sign > 0 else 2**31
    print(f"dấu = {sign:+d}, |x| = {x}, giới hạn độ lớn = {limit}")

    rev = 0
    step = 1
    while x != 0:
        digit = x % 10
        x //= 10
        if rev > limit // 10 or (rev == limit // 10 and digit > limit % 10):
            print(f"  bước {step:2d}: digit = {digit}, rev = {rev} -> rev*10 + digit sẽ vượt {limit}. Trả về 0")
            return 0
        rev = rev * 10 + digit
        print(f"  bước {step:2d}: digit = {digit}, x còn {x:>10}, rev = {rev}")
        step += 1

    print(f"  kết quả = {sign} * {rev} = {sign * rev}")
    return sign * rev

trace_reverse(-120)
print()
trace_reverse(1534236469)

dấu = -1, |x| = 120, giới hạn độ lớn = 2147483648
  bước  1: digit = 0, x còn         12, rev = 0
  bước  2: digit = 2, x còn          1, rev = 2
  bước  3: digit = 1, x còn          0, rev = 21
  kết quả = -1 * 21 = -21

dấu = +1, |x| = 1534236469, giới hạn độ lớn = 2147483647
  bước  1: digit = 9, x còn  153423646, rev = 9
  bước  2: digit = 6, x còn   15342364, rev = 96
  bước  3: digit = 4, x còn    1534236, rev = 964
  bước  4: digit = 6, x còn     153423, rev = 9646
  bước  5: digit = 3, x còn      15342, rev = 96463
  bước  6: digit = 2, x còn       1534, rev = 964632
  bước  7: digit = 4, x còn        153, rev = 9646324
  bước  8: digit = 3, x còn         15, rev = 96463243
  bước  9: digit = 5, x còn          1, rev = 964632435
  bước 10: digit = 1, rev = 964632435 -> rev*10 + digit sẽ vượt 2147483647. Trả về 0


0

# 10. Phiên bản đúng format LeetCode

In [16]:
class Solution:
    def reverse(self, x: int) -> int:
        INT_MAX = 2**31 - 1
        sign = -1 if x < 0 else 1
        x = abs(x)
        limit = INT_MAX if sign > 0 else INT_MAX + 1

        rev = 0
        while x:
            digit = x % 10
            x //= 10
            if rev > limit // 10 or (rev == limit // 10 and digit > limit % 10):
                return 0
            rev = rev * 10 + digit

        return sign * rev

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<34} PASS  {actual}")


sol = Solution()
check("Ví dụ 1: 123", sol.reverse(123), 321)
check("Ví dụ 2: -123", sol.reverse(-123), -321)
check("Ví dụ 3: 120", sol.reverse(120), 21)
check("Số 0", sol.reverse(0), 0)
check("Một chữ số âm", sol.reverse(-5), -5)
check("Nhiều số 0 ở cuối", sol.reverse(-1200), -21)
check("Tràn dương", sol.reverse(1534236469), 0)
check("INT_MAX -> tràn", sol.reverse(2**31 - 1), 0)
check("INT_MIN -> tràn", sol.reverse(-2**31), 0)
check("Sát INT_MAX nhưng không tràn", sol.reverse(1463847412), 2147483641)
check("Sát INT_MIN nhưng không tràn", sol.reverse(-1463847412), -2147483641)
check("Tràn vì chữ số đầu lớn", sol.reverse(1000000003), 0)

Ví dụ 1: 123                       PASS  321
Ví dụ 2: -123                      PASS  -321
Ví dụ 3: 120                       PASS  21
Số 0                               PASS  0
Một chữ số âm                      PASS  -5
Nhiều số 0 ở cuối                  PASS  -21
Tràn dương                         PASS  0
INT_MAX -> tràn                    PASS  0
INT_MIN -> tràn                    PASS  0
Sát INT_MAX nhưng không tràn       PASS  2147483641
Sát INT_MIN nhưng không tràn       PASS  -2147483641
Tràn vì chữ số đầu lớn             PASS  0


## Đối chiếu với cách dùng chuỗi

In [18]:
import random

random.seed(7)
samples = [random.randint(INT_MIN, INT_MAX) for _ in range(20000)]
samples += [INT_MIN, INT_MAX, 0, 1, -1, 10, -10]

for x in samples:
    assert sol.reverse(x) == reverse_string(x), x

overflow_count = sum(1 for x in samples if sol.reverse(x) == 0 and x != 0)
print(f"{len(samples)} test: hai cách khớp nhau ({overflow_count} trường hợp tràn số)")

20007 test: hai cách khớp nhau (8432 trường hợp tràn số)


# 11. Vì sao thuật toán đúng?

**Bất biến:** sau `k` lần lặp, `rev` là số tạo bởi `k` chữ số cuối của `|x|` (ban đầu) viết theo thứ tự ngược, còn `x` là phần chưa xử lý.

- Mỗi vòng bóc đúng một chữ số cuối (`% 10`) và bỏ nó khỏi `x` (`// 10`).
- `rev * 10 + digit` gắn chữ số đó vào cuối `rev`.
- Khi `x == 0`, mọi chữ số đã được đảo.

**Kiểm tra tràn chính xác:** theo bảng ở mục 7, điều kiện `rev > limit // 10 or (rev == limit // 10 and digit > limit % 10)` đúng **khi và chỉ khi** `rev * 10 + digit > limit`. Và vì `rev` chỉ tăng qua mỗi bước, một khi đã vượt giới hạn thì không thể quay lại — dừng ngay là đúng.

**Dấu:** đảo chữ số không phụ thuộc dấu, nên `reverse(-n) = -reverse(n)`; phía âm cho phép độ lớn tới `2^31`.

# 12. Độ phức tạp

Gọi `d` là số chữ số của `x`. Mỗi vòng lặp bỏ một chữ số, nên có `d` vòng.

Vì `|x| ≤ 2^31`, ta có `d ≤ 10`.

```text
Time complexity:  O(d) = O(log10 |x|)  — tối đa 10 vòng, thực chất là O(1)
Space complexity: O(1)
```

Cách dùng chuỗi cũng `O(d)` thời gian, nhưng tạo thêm chuỗi dài `d` (bộ nhớ `O(d)`), và quan trọng hơn là **tạo ra số có thể vượt 32-bit**.

# 13. Những lỗi dễ mắc

## Lỗi 1: Dùng `%` và `//` trực tiếp trên số âm

In [19]:
def wrong_negative(x: int) -> int:
    rev = 0
    steps = 0
    while x != 0 and steps < 6:          # giới hạn số bước để demo không bị treo
        digit = x % 10
        x //= 10
        rev = rev * 10 + digit
        steps += 1
        print(f"  digit = {digit}, x = {x}, rev = {rev}")
    return rev

print("Sai với -123:")
wrong_negative(-123)
print("Đúng:", reverse_integer(-123))

Sai với -123:
  digit = 7, x = -13, rev = 7
  digit = 7, x = -2, rev = 77
  digit = 8, x = -1, rev = 778
  digit = 9, x = -1, rev = 7789
  digit = 9, x = -1, rev = 77899
  digit = 9, x = -1, rev = 778999
Đúng: -321


Với `-123`: `digit` đầu tiên là `7` (không phải `3`), và `x` tiến tới `-1` rồi kẹt ở đó mãi vì `-1 // 10 == -1`. Nếu không có giới hạn `steps`, vòng lặp chạy vô hạn.

## Lỗi 2: Kiểm tra tràn sau khi đã tính

```python
rev = rev * 10 + digit
if rev > INT_MAX:
    return 0
```

Trong Python, code này cho đáp án đúng vì Python không tràn. Nhưng trong môi trường 32-bit thật, `rev * 10` đã **tràn và quay vòng** (wrap around) thành một số rác, phép so sánh sau đó vô nghĩa.

Ta có thể mô phỏng điều C/Java sẽ làm:

In [20]:
def to_int32(n: int) -> int:
    """Mô phỏng tràn số 32-bit: giữ 32 bit thấp, diễn giải theo bù 2."""
    n &= 0xFFFFFFFF
    return n - 2**32 if n >= 2**31 else n

rev = 964632435
print("Python tính đúng:   ", rev * 10 + 1)
print("Máy 32-bit thấy:    ", to_int32(rev * 10 + 1))
print("Có lớn hơn INT_MAX? ", to_int32(rev * 10 + 1) > INT_MAX)

Python tính đúng:    9646324351
Máy 32-bit thấy:     1056389759
Có lớn hơn INT_MAX?  False


Máy 32-bit thấy một số "rác" `1056389759` — trông hoàn toàn hợp lệ, nằm trong phạm vi — nên phép kiểm tra `> INT_MAX` trả về `False`. Lỗi lọt qua, và hàm trả về một kết quả sai thay vì `0`!

## Lỗi 3: Dùng `/` thay vì `//`

`123 / 10 = 12.3` (số thực). Vòng lặp `while x != 0` gần như không bao giờ dừng đúng, và `digit` cũng thành số thực.

## Lỗi 4: Đặt dấu sai chỗ khi dùng chuỗi

`str(-123)[::-1]` cho `"321-"` — `int("321-")` báo lỗi. Phải lấy `abs(x)` trước khi đổi sang chuỗi.

In [21]:
print(str(-123)[::-1])
try:
    int(str(-123)[::-1])
except ValueError as e:
    print("Lỗi:", e)

321-
Lỗi: invalid literal for int() with base 10: '321-'


## Lỗi 5: Quên giới hạn phía âm khác phía dương

Phía âm cho phép độ lớn `2147483648`, phía dương chỉ `2147483647`. Dùng chung `INT_MAX` cho cả hai thì về lý thuyết sẽ từ chối nhầm `-2147483648`.

Trên thực tế với bài này, trường hợp `rev == 214748364` chỉ xảy ra ở chữ số cuối cùng, và chữ số đó chính là **chữ số đầu** của `x` (với `x` có 10 chữ số, chữ số đầu chỉ có thể là `1` hoặc `2`). Nên phép kiểm tra `digit > 7` hay `digit > 8` không tạo khác biệt về kết quả ở đây.

Dù vậy, hãy viết đúng giới hạn cho từng dấu: logic rõ ràng hơn, và khi bạn dùng lại đoạn chống tràn cho bài khác (ví dụ một hàm phải **báo lỗi** khi tràn), giới hạn sai sẽ từ chối nhầm đúng giá trị `-2147483648`.

# 14. Mẹo nhận diện pattern

**Thao tác chữ số (digit manipulation):**

```python
while x:
    digit = x % 10      # bóc chữ số cuối
    x //= 10            # bỏ chữ số cuối
    rev = rev * 10 + digit   # đẩy vào cuối kết quả
```

Gặp ở: **#9 Palindrome Number**, **#258 Add Digits**, **#202 Happy Number**, **#2 Add Two Numbers** (tách chữ số và số nhớ).

**Chống tràn trước khi nhân:**

```python
if rev > LIMIT // 10 or (rev == LIMIT // 10 and digit > LIMIT % 10):
    # tràn
```

Gặp ở: **#8 String to Integer (atoi)** (bài medium kế tiếp!), **#29 Divide Two Integers**, các bài "parse số".

**Dấu hiệu nhận ra:** đề nhắc tới "32-bit signed integer", "return 0 if overflow", "assume environment does not allow 64-bit".

# 15. Biến thể: đảo số ở cơ số bất kỳ

Thay `10` bằng `base`. Ví dụ ở cơ số 2: `6` là `110`, đảo thành `011` = `3`.

In [22]:
def reverse_in_base(x: int, base: int) -> int:
    rev = 0
    while x:
        x, digit = divmod(x, base)
        rev = rev * base + digit
    return rev

print(reverse_in_base(6, 2))     # 110 -> 011 = 3
print(reverse_in_base(123, 10))  # 321
print(bin(13), "->", bin(reverse_in_base(13, 2)))

3
321
0b1101 -> 0b1011


# 16. Template tư duy

```python
INT_MAX = 2**31 - 1
sign = -1 if x < 0 else 1
x = abs(x)
limit = INT_MAX if sign > 0 else INT_MAX + 1

rev = 0
while x:
    digit = x % 10
    x //= 10
    if rev > limit // 10 or (rev == limit // 10 and digit > limit % 10):
        return 0
    rev = rev * 10 + digit

return sign * rev
```

Ba điều cần nhớ: **tách dấu trước**, **kiểm tra tràn trước khi nhân**, **phía âm rộng hơn 1**.

# 17. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Không chạy code. Lập bảng `digit`, `x`, `rev` cho `x = -1200`. Kết quả là gì?

## Bài 2 — Đoán output

```python
print(-7 // 2)
print(-7 % 2)
print(7 // -2)
print(2147483647 // 10, 2147483647 % 10)
print(int("0021"))
```

## Bài 3 — Đếm số chữ số

Viết hàm đếm số chữ số của một số nguyên (bỏ qua dấu) **bằng toán**, không dùng `str`.

```python
count_digits(12345) -> 5
count_digits(-90)   -> 2
count_digits(0)     -> 1
```

In [23]:
def exercise_3_count_digits(x: int) -> int:
    # TODO:
    # Xử lý riêng x == 0
    # Lấy abs(x), lặp // 10 cho tới khi bằng 0, đếm số lần
    pass


# assert exercise_3_count_digits(12345) == 5
# assert exercise_3_count_digits(-90) == 2
# assert exercise_3_count_digits(0) == 1

## Bài 4 — Điền chỗ trống: lời giải chính

In [24]:
def exercise_4_reverse(x: int) -> int:
    INT_MAX = 2**31 - 1
    # TODO 1: tách dấu, lấy x = abs(x), tính limit
    rev = 0
    # TODO 2: vòng lặp bóc chữ số
    # TODO 3: kiểm tra tràn TRƯỚC khi rev = rev * 10 + digit
    # TODO 4: trả về sign * rev
    pass


# assert exercise_4_reverse(123) == 321
# assert exercise_4_reverse(-120) == -21
# assert exercise_4_reverse(1534236469) == 0

## Bài 5 — Biến thể: Palindrome Number bằng cách đảo số

Dùng ý tưởng đảo số để kiểm tra một số nguyên có phải palindrome không (số âm không phải palindrome).

```python
is_palindrome_number(121)  -> True
is_palindrome_number(-121) -> False
is_palindrome_number(10)   -> False
```

In [25]:
def exercise_5_is_palindrome_number(x: int) -> bool:
    # TODO
    pass


# assert exercise_5_is_palindrome_number(121) is True
# assert exercise_5_is_palindrome_number(-121) is False
# assert exercise_5_is_palindrome_number(10) is False
# assert exercise_5_is_palindrome_number(0) is True

## Bài 6 — Mini challenge: đọc chuỗi chữ số thành số 32-bit (khởi động cho bài #8)

Cho chuỗi `s` gồm một dấu tùy chọn (`'+'` hoặc `'-'`) theo sau là **chỉ các chữ số**. Đổi `s` thành số nguyên, nhưng nếu vượt phạm vi 32-bit thì **kẹp (clamp)** về `INT_MAX` hoặc `INT_MIN`.

Yêu cầu: **không** dùng `int(s)`; xử lý từng ký tự và kiểm tra tràn trước khi nhân.

```python
parse_int32("42")           -> 42
parse_int32("-0042")        -> -42
parse_int32("2147483648")   -> 2147483647    (kẹp về INT_MAX)
parse_int32("-2147483648")  -> -2147483648   (vừa đúng INT_MIN)
parse_int32("-99999999999") -> -2147483648
```

Gợi ý nhanh: `ord(c) - ord('0')` đổi ký tự chữ số thành số.

In [26]:
def exercise_6_parse_int32(s: str) -> int:
    # TODO
    pass


# assert exercise_6_parse_int32("42") == 42
# assert exercise_6_parse_int32("-0042") == -42
# assert exercise_6_parse_int32("2147483648") == 2147483647
# assert exercise_6_parse_int32("-2147483648") == -2147483648
# assert exercise_6_parse_int32("-99999999999") == -2147483648

# 18. Gợi ý

**Bài 1**
- Mức 1: tách dấu trước, làm với `1200`.
- Mức 2: hai chữ số đầu bóc ra đều là `0`, `rev` vẫn là `0`.
- Mức 3: cuối cùng `rev = 21`, gắn dấu.

**Bài 2**
- Mức 1: `//` làm tròn **xuống** (về âm vô cùng).
- Mức 2: `%` có cùng dấu với **số chia**.
- Mức 3: kiểm tra bằng đẳng thức `(a // b) * b + a % b == a`.

**Bài 3**
- Mức 1: giống vòng `while x` của lời giải chính, nhưng chỉ đếm.
- Mức 2: `if x == 0: return 1`.
- Mức 3: `x = abs(x); count = 0; while x: x //= 10; count += 1`.

**Bài 4**
- Mức 1: so với template mục 16.
- Mức 2: `limit = INT_MAX if sign > 0 else INT_MAX + 1`.
- Mức 3: `if rev > limit // 10 or (rev == limit // 10 and digit > limit % 10): return 0`.

**Bài 5**
- Mức 1: số âm → `False` ngay.
- Mức 2: đảo `x` (không cần lo tràn trong Python) rồi so với `x` ban đầu.
- Mức 3: nhớ lưu bản sao `original = x` trước khi vòng lặp làm `x` về 0.

**Bài 6**
- Mức 1: xử lý ký tự dấu ở `s[0]`, đặt `limit` giống bài này.
- Mức 2: duyệt từng ký tự chữ số, `digit = ord(c) - ord('0')`.
- Mức 3: khi phát hiện tràn, **không** trả về 0 mà trả về `INT_MAX` (nếu dương) hoặc `INT_MIN` (nếu âm).

# 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

`sign = -1`, làm với `x = 1200`:

| Bước | digit | x sau | rev |
|---|---|---|---|
| 1 | 0 | 120 | 0 |
| 2 | 0 | 12 | 0 |
| 3 | 2 | 1 | 2 |
| 4 | 1 | 0 | 21 |

Kết quả `-1 * 21 = -21`.

In [27]:
trace_reverse(-1200)

dấu = -1, |x| = 1200, giới hạn độ lớn = 2147483648
  bước  1: digit = 0, x còn        120, rev = 0
  bước  2: digit = 0, x còn         12, rev = 0
  bước  3: digit = 2, x còn          1, rev = 2
  bước  4: digit = 1, x còn          0, rev = 21
  kết quả = -1 * 21 = -21


-21

## Đáp án Bài 2

In [28]:
print(-7 // 2)                            # -4  (làm tròn xuống từ -3.5)
print(-7 % 2)                             # 1   (vì -4 * 2 + 1 = -7)
print(7 // -2)                            # -4
print(2147483647 // 10, 2147483647 % 10)  # 214748364 7
print(int("0021"))                        # 21

-4
1
-4
214748364 7
21


## Đáp án Bài 3

In [29]:
def answer_3_count_digits(x: int) -> int:
    if x == 0:
        return 1
    x = abs(x)
    count = 0
    while x:
        x //= 10
        count += 1
    return count

assert answer_3_count_digits(12345) == 5
assert answer_3_count_digits(-90) == 2
assert answer_3_count_digits(0) == 1
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [30]:
def answer_4_reverse(x: int) -> int:
    INT_MAX = 2**31 - 1
    sign = -1 if x < 0 else 1
    x = abs(x)
    limit = INT_MAX if sign > 0 else INT_MAX + 1
    rev = 0
    while x:
        digit = x % 10
        x //= 10
        if rev > limit // 10 or (rev == limit // 10 and digit > limit % 10):
            return 0
        rev = rev * 10 + digit
    return sign * rev

assert answer_4_reverse(123) == 321
assert answer_4_reverse(-120) == -21
assert answer_4_reverse(1534236469) == 0
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [31]:
def answer_5_is_palindrome_number(x: int) -> bool:
    if x < 0:
        return False
    original = x
    rev = 0
    while x:
        rev = rev * 10 + x % 10
        x //= 10
    return rev == original

assert answer_5_is_palindrome_number(121) is True
assert answer_5_is_palindrome_number(-121) is False
assert answer_5_is_palindrome_number(10) is False
assert answer_5_is_palindrome_number(0) is True
print("Bài 5 PASS")

Bài 5 PASS


## Đáp án Bài 6

Cấu trúc gần như giống hệt `reverse`, chỉ khác: chữ số lấy từ **chuỗi** (trái sang phải) thay vì bóc từ số, và khi tràn thì **kẹp** thay vì trả về 0.

In [32]:
def answer_6_parse_int32(s: str) -> int:
    INT_MAX, INT_MIN = 2**31 - 1, -2**31
    sign = 1
    i = 0
    if s and s[0] in "+-":
        sign = -1 if s[0] == "-" else 1
        i = 1
    limit = INT_MAX if sign > 0 else INT_MAX + 1

    value = 0
    for c in s[i:]:
        digit = ord(c) - ord("0")
        if value > limit // 10 or (value == limit // 10 and digit > limit % 10):
            return INT_MAX if sign > 0 else INT_MIN
        value = value * 10 + digit
    return sign * value

assert answer_6_parse_int32("42") == 42
assert answer_6_parse_int32("-0042") == -42
assert answer_6_parse_int32("2147483648") == 2147483647
assert answer_6_parse_int32("-2147483648") == -2147483648
assert answer_6_parse_int32("-99999999999") == -2147483648
print("Bài 6 PASS")

Bài 6 PASS


Để ý: với `"-2147483648"`, `value` đạt `214748364` rồi gặp `digit = 8`.

- Dùng `limit` riêng cho phía âm (`digit > 8`?): không tràn, tính tiếp ra đúng `-2147483648`.
- Nếu dùng `INT_MAX` cho cả phía âm (`digit > 7`?): bị coi là tràn và **kẹp** về `INT_MIN` — tình cờ vẫn ra `-2147483648`.

Ở bài này kết quả giống nhau nhờ kẹp. Nhưng nếu đề yêu cầu *trả 0* hay *báo lỗi* khi tràn, cách thứ hai sẽ sai đúng ở giá trị biên này. Vì vậy giữ `limit` riêng cho từng dấu.

# 20. Tổng kết

Cốt lõi của bài:

```python
digit = x % 10
x //= 10
rev = rev * 10 + digit
```

cộng thêm **một dòng chống tràn** đặt **trước** phép nhân:

```python
if rev > limit // 10 or (rev == limit // 10 and digit > limit % 10):
    return 0
```

---

## Checklist trước khi nộp

- [ ] Tách dấu, làm việc với `abs(x)` (tránh bẫy `%`, `//` với số âm)?
- [ ] Dùng `//`, không dùng `/`?
- [ ] Kiểm tra tràn **trước** khi tính `rev * 10 + digit`?
- [ ] Giới hạn phía âm là `2^31`, phía dương là `2^31 - 1`?
- [ ] Đã thử `0`, `120`, `-123`, `1534236469`, `INT_MIN`, `INT_MAX`?

---

## Khi nào nên nghĩ tới kỹ thuật này?

- Đề nói "**32-bit signed integer**", "return 0 / clamp if overflow"
- Đề cấm dùng 64-bit, hoặc yêu cầu không đổi sang chuỗi
- Cần **bóc chữ số** của một số (đảo, tính tổng chữ số, kiểm tra palindrome)